# 5.01 VAE 与 CVAE：用隐变量表示"多种做法"

[![在 Colab 中打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ROI-su7/vla-ladder/blob/main/notebooks/05_act/5.01_vae_cvae.ipynb)


## 为什么要学

第 3 章看到：同一个状态有多种正确动作时，MSE 会取平均。CVAE（条件变分自编码器）的思路是引入一个**隐变量 $z$**，比如"这次从左边绕"还是"从右边绕"。给定状态和 $z$，动作就是确定的（单峰的）；不同的 $z$ 对应不同的模式。ACT 就是一个 CVAE。

本课从 VAE 的数学推导开始，在一个一维双分支数据上实现 CVAE，并观察一个在 ACT 中真实存在的现象：**后验坍缩**。

---

## 1. 隐变量模型

假设数据 $y$ 由一个看不见的隐变量 $z$ 生成：

$$
p_\theta(y)=\int p_\theta(y|z)\,p(z)\,dz,\qquad p(z)=\mathcal N(0, I)
$$

- $p(z)$：先验，简单的标准高斯；
- $p_\theta(y|z)$：解码器（神经网络），给定 $z$ 生成 $y$。

问题：这个积分算不出来，没法直接做最大似然。

## 2. 证据下界（ELBO）

引入一个**编码器** $q_\phi(z|y)$ 近似真实后验 $p(z|y)$，可以推出（用 Jensen 不等式或 KL 的非负性）：

$$
\log p_\theta(y)\ \ge\ \underbrace{\mathbb E_{z\sim q_\phi(z|y)}\big[\log p_\theta(y|z)\big]}_{\text{重建项}}\ -\ \underbrace{D_{KL}\big(q_\phi(z|y)\,\|\,p(z)\big)}_{\text{正则项}}\ =\ \text{ELBO}
$$

推导（一行）：$\log p(y)=\text{ELBO}+D_{KL}(q_\phi(z|y)\|p(z|y))\ge\text{ELBO}$，因为 KL 非负。

最大化 ELBO：

- **重建项**：编码器把 $y$ 编码成 $z$，解码器能从 $z$ 还原 $y$。若 $p_\theta(y|z)$ 是固定方差的高斯，它就是 **MSE**（ACT 用 L1，对应拉普拉斯分布）；
- **KL 项**：编码出的 $z$ 分布要接近先验 $\mathcal N(0,I)$。这样**推理时从先验采样的 $z$ 才能解码出合理的 $y$**。

编码器输出均值 $\mu$ 和对数方差 $\log\sigma^2$，KL 用 1.05 课的闭式解；采样用重参数化 $z=\mu+\sigma\epsilon$。

## 3. 条件 VAE（CVAE）

在每个分布上都加上条件 $x$（对机器人来说，$x$ 是观测，$y$ 是动作块）：

$$
\text{ELBO}(x,y)=\mathbb E_{q_\phi(z|x,y)}\big[\log p_\theta(y|x,z)\big]-D_{KL}\big(q_\phi(z|x,y)\,\|\,p(z)\big)
$$

- 训练时：编码器看到**真实动作** $y$，推断"这条示教用的是哪种模式" $z$；解码器根据 $(x,z)$ 重建动作。
- 推理时：没有真实动作，编码器扔掉，从先验 $p(z)$ 采样 $z$（或者像 ACT 那样直接取 $z=0$），解码器输出动作。

---

## 4. 实验：一维双分支数据

$x\sim U(-1,1)$，$y$ 以各 50% 的概率取上分支 $0.3+0.5x^2$ 或下分支 $-0.3-0.5x^2$（再加一点噪声）。

In [ ]:
# 🔧 环境准备（每个 notebook 都先运行这一格）
# - 在 Colab 中：自动下载本项目代码
# - 在本地：自动把项目根目录加入 sys.path（也可以在根目录执行 `pip install -e .`）
import os, sys
if "google.colab" in sys.modules:
    if not os.path.exists("/content/vla-ladder"):
        os.system("git clone -q --depth 1 https://github.com/ROI-su7/vla-ladder.git /content/vla-ladder")
    sys.path.insert(0, "/content/vla-ladder")
else:
    _p = os.path.abspath(".")
    while _p != os.path.dirname(_p) and not os.path.isdir(os.path.join(_p, "vla_ladder")):
        _p = os.path.dirname(_p)
    sys.path.insert(0, _p)
print("环境就绪 ✅")

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from vla_ladder.utils import set_seed

set_seed(0)
N = 4000
x = torch.rand(N, 1) * 2 - 1
branch = torch.where(torch.rand(N, 1) < 0.5, 1.0, -1.0)
y = branch * (0.3 + 0.5 * x ** 2) + 0.03 * torch.randn(N, 1)

def mlp(i, o, h=128):
    return nn.Sequential(nn.Linear(i, h), nn.ReLU(), nn.Linear(h, h), nn.ReLU(), nn.Linear(h, o))

reg = mlp(1, 1); opt = torch.optim.Adam(reg.parameters(), 1e-3)
for _ in range(2000):
    loss = F.mse_loss(reg(x), y); opt.zero_grad(); loss.backward(); opt.step()
xs = torch.linspace(-1, 1, 200)[:, None]
plt.figure(figsize=(4.5, 3.5)); plt.scatter(x, y, s=1, alpha=0.3, label="data")
plt.plot(xs, reg(xs).detach(), "r", lw=2, label="MSE regression"); plt.legend(); plt.title("MSE learns the average"); plt.show()

### 练习 1：CVAE

In [ ]:
class CVAE(nn.Module):
    def __init__(self, x_dim=1, y_dim=1, z_dim=2, h=128):
        super().__init__()
        self.z_dim = z_dim
        self.encoder = mlp(x_dim + y_dim, 2 * z_dim, h)       # q(z | x, y) -> (mu, logvar)
        self.decoder = mlp(x_dim + z_dim, y_dim, h)           # p(y | x, z) 的均值

    def loss(self, x, y, beta=1.0):
        """返回 (总损失, 重建项, KL 项)"""
        # TODO:
        # 1) mu, logvar = encoder(cat[x, y]).chunk(2, -1)
        # 2) z = mu + exp(0.5*logvar) * randn_like(mu)       （重参数化）
        # 3) recon = MSE(decoder(cat[x, z]), y)
        # 4) kl = 0.5 * sum(mu² + exp(logvar) - 1 - logvar, dim=-1).mean()
        # 5) return recon + beta * kl, recon, kl
        raise NotImplementedError

    @torch.no_grad()
    def sample(self, x):
        """推理：z ~ N(0, I)"""
        # TODO
        raise NotImplementedError

In [ ]:
#@title 💡 参考答案（点击展开查看；运行本单元即可使用参考实现） { display-mode: "form" }
class CVAE(nn.Module):
    def __init__(self, x_dim=1, y_dim=1, z_dim=2, h=128):
        super().__init__()
        self.z_dim = z_dim
        self.encoder = mlp(x_dim + y_dim, 2 * z_dim, h)
        self.decoder = mlp(x_dim + z_dim, y_dim, h)

    def loss(self, x, y, beta=1.0):
        """返回 (总损失, 重建项, KL 项)"""
        mu, logvar = self.encoder(torch.cat([x, y], -1)).chunk(2, -1)
        z = mu + (0.5 * logvar).exp() * torch.randn_like(mu)
        recon = F.mse_loss(self.decoder(torch.cat([x, z], -1)), y)
        kl = 0.5 * (mu.pow(2) + logvar.exp() - 1 - logvar).sum(-1).mean()
        return recon + beta * kl, recon, kl

    @torch.no_grad()
    def sample(self, x):
        """推理：z ~ N(0, I)"""
        z = torch.randn(x.shape[0], self.z_dim)
        return self.decoder(torch.cat([x, z], -1))

In [ ]:
# ✅ 自测：训练并从先验采样
def train_cvae(beta, steps=3000, seed=0):
    set_seed(seed)
    m = CVAE(); opt = torch.optim.Adam(m.parameters(), 1e-3)
    for _ in range(steps):
        i = torch.randint(0, N, (256,))
        loss, rec, kl = m.loss(x[i], y[i], beta)
        opt.zero_grad(); loss.backward(); opt.step()
    return m, rec.item(), kl.item()

cvae, rec, kl = train_cvae(beta=0.01)
xt = torch.rand(2000, 1) * 2 - 1
ys = cvae.sample(xt)
frac_upper = (ys > 0).float().mean().item()
near_branch = ((ys.abs() - (0.3 + 0.5 * xt ** 2)).abs() < 0.1).float().mean().item()
print(f"β=0.01: 重建 {rec:.4f}  KL {kl:.3f}  采样落在上分支的比例 {frac_upper:.2f}，落在某个分支附近的比例 {near_branch:.2f}")
assert 0.3 < frac_upper < 0.7 and near_branch > 0.8
print("通过 ✅")

---

## 5. β 的影响与后验坍缩

把 ELBO 写成 $\text{重建} + \beta\cdot\text{KL}$（$\beta$-VAE）：

- $\beta$ 小：$z$ 可以携带很多信息，重建好，但先验采样可能落到"没见过"的区域；
- $\beta$ 大：KL 惩罚太重，编码器干脆输出 $\mu=0,\sigma=1$，让 KL=0，**解码器完全忽略 $z$**。这叫**后验坍缩**（posterior collapse），CVAE 退化成普通回归，又回到"取平均"。

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
for ax, beta in zip(axes, [0.01, 0.1, 3.0]):
    m, rec, kl = train_cvae(beta)
    ax.scatter(x, y, s=1, alpha=0.15, c="gray")
    ax.scatter(xt, m.sample(xt), s=2, c="C1")
    ax.set_title(f"beta={beta}: recon={rec:.3f}, KL={kl:.2f}")
plt.tight_layout(); plt.show()

观察：

- $\beta=3$ 时 KL 几乎为 0，采样结果全部挤在中间：**$z$ 没被使用**；
- $\beta=0.1$ 时 KL 约为 0.7~0.8，非常接近下面要说的"编码 1 比特信息的代价" $\ln 2\approx0.69$。模型恰好只用 $z$ 记住了"走哪个分支"这 1 比特，别的什么都不存。

**判断 KL 是否值得付出**：编码"走哪个分支"这 1 比特信息，至少要付出约 $\ln 2\approx0.69$ 的 KL。如果 $\beta\times0.69$ 大于它能减少的重建误差，优化器就会选择坍缩。下一课会看到：**ACT 默认的 $\beta=10$ 在我们的任务上就会坍缩**，但 ACT 仍然能工作，原因很有意思。

### 练习 2：让编码器"告诉我们"它学到了什么

对训练好的 β=0.01 模型，把上分支和下分支的数据分别送进编码器，看看它们的 $\mu$ 是否被分开了（即 $z$ 是否编码了"哪个分支"）。

In [ ]:
@torch.no_grad()
def latent_means(model, x, y):
    # TODO: 返回编码器输出的 mu，形状 (N, z_dim)
    raise NotImplementedError

In [ ]:
#@title 💡 参考答案（点击展开查看；运行本单元即可使用参考实现） { display-mode: "form" }
@torch.no_grad()
def latent_means(model, x, y):
    mu, _ = model.encoder(torch.cat([x, y], -1)).chunk(2, -1)
    return mu

In [ ]:
# ✅ 自测
mu = latent_means(cvae, x, y)
up, down = mu[branch[:, 0] > 0], mu[branch[:, 0] < 0]
sep = (up.mean(0) - down.mean(0)).norm().item()
plt.figure(figsize=(4, 4)); plt.scatter(up[:, 0], up[:, 1], s=2, label="upper branch"); plt.scatter(down[:, 0], down[:, 1], s=2, label="lower branch")
plt.legend(); plt.title("encoder means q(z|x,y)"); plt.xlabel("z1"); plt.ylabel("z2"); plt.show()
print(f"两个分支的隐变量均值之间的距离 {sep:.2f}")
assert sep > 1.0
print("通过 ✅ z 编码了'走哪个分支'")

---

## 小结

- VAE：用编码器 $q(z|y)$ 近似后验，最大化 ELBO = 重建 − KL；重参数化让采样可导。
- CVAE：所有分布都以观测 $x$ 为条件；训练时编码器看真实动作，推理时从先验采 $z$。
- $\beta$ 太大会后验坍缩，$z$ 被忽略，CVAE 退化为普通回归。
- 下一课：ACT = CVAE（编码器、解码器都是 Transformer）+ 动作分块 + 时间集成。